## Load Data and Drop Channel

In [80]:
import numpy as np

data = np.load(
    "Datasets/Main.npz",
    allow_pickle=True
)

X = data["X_raw"]
y = data["y"]

print("Original Dataset")
print("X:", X.shape)
print("y:", y.shape)

BAD_CHANNELS = [0]

X = np.delete(
    X,
    BAD_CHANNELS,
    axis=1
)

print("\nAfter Channel Removal")
print("X:", X.shape)

unique, counts = np.unique(
    y,
    return_counts=True
)

print("\nLabel Distribution")

for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count}")



print("\nFinal Dataset")
print("X:", X.shape)
print("y:", y.shape)

Original Dataset
X: (570, 8, 1750)
y: (570,)

After Channel Removal
X: (570, 7, 1750)

Label Distribution
Class 1: 285
Class 2: 285

Final Dataset
X: (570, 7, 1750)
y: (570,)


In [76]:
import numpy as np

baseline_samples = 500

snr = np.zeros((X.shape[0], X.shape[1]))
snr_db = np.zeros_like(snr)

for trial in range(X.shape[0]):
    for ch in range(X.shape[1]):

        signal = X[trial, ch]

        # First 500 samples = baseline
        baseline = signal[:baseline_samples]

        # Noise estimate
        noise_std = np.std(baseline, ddof=1)

        # Peak response after baseline
        peak = np.max(np.abs(signal[baseline_samples:]))

        snr[trial, ch] = peak / noise_std

        if noise_std > 0:
            snr_db[trial, ch] = 20 * np.log10(snr[trial, ch])
        else:
            snr_db[trial, ch] = np.nan

print("Mean SNR:", np.mean(snr))
print("Mean SNR (dB):", np.mean(snr_db))

print("Per-channel mean SNR:", np.mean(snr, axis=0))
print("Per-channel mean SNR (dB):", np.mean(snr_db, axis=0))

Mean SNR: 3.203041707569591
Mean SNR (dB): 9.158517548746822
Per-channel mean SNR: [3.11546467 3.16246154 3.33119892]
Per-channel mean SNR (dB): [8.90908133 9.17501366 9.39145766]


## Signal Processing

In [81]:
import numpy as np
from brainflow.data_filter import (
    DataFilter,
    FilterTypes,
    DetrendOperations
)

SAMPLING_RATE = 250  
LOW_CUT = 8
HIGH_CUT = 13
FILTER_ORDER = 4

X_filtered = np.copy(X)

for trial in range(X.shape[0]):

    for ch in range(X.shape[1]):

        signal = X_filtered[trial, ch]
        signal = np.ascontiguousarray(
            signal,
            dtype=np.float64
        )

        DataFilter.detrend(
            signal,
            DetrendOperations.CONSTANT.value
        )

        DataFilter.perform_bandpass(
            signal,
            SAMPLING_RATE,
            LOW_CUT,
            HIGH_CUT,
            FILTER_ORDER,
            FilterTypes.BUTTERWORTH_ZERO_PHASE.value,
            0
        )

        DataFilter.remove_environmental_noise(
            signal,
            SAMPLING_RATE,
            1
        )

        X_filtered[trial, ch] = signal

print("Original Shape :", X.shape)
print("Filtered Shape :", X_filtered.shape)

X = X_filtered

Original Shape : (570, 7, 1750)
Filtered Shape : (570, 7, 1750)


## ONLY FOR SELF DATA

In [95]:
X = X[:, :, 500:-75]
print("X shape:", X.shape)

X shape: (90, 3, 1175)


## Noisy Trial Rejection

In [75]:
import numpy as np

def reject_noisy_trials(
    X,
    y,
    max_abs_thresh=250,
    ptp_thresh=500,
    std_z_thresh=3.0,
    flat_std_thresh=1e-6,
    verbose=True
):
    """
    X shape: (trials, channels, samples)
    y shape: (trials,)
    """

    # =========================
    # TRIAL / CHANNEL STATS
    # =========================

    max_abs = np.max(np.abs(X), axis=2)    # (trials, channels)
    ptp = np.ptp(X, axis=2)                # (trials, channels)
    std = np.std(X, axis=2)                # (trials, channels)

    trial_max_abs = np.max(max_abs, axis=1)
    trial_ptp = np.max(ptp, axis=1)
    trial_mean_std = np.mean(std, axis=1)

    median_std = np.median(trial_mean_std)
    mad_std = np.median(np.abs(trial_mean_std - median_std)) + 1e-12
    robust_z_std = 0.6745 * (trial_mean_std - median_std) / mad_std

    # =========================
    # REJECTION REASONS
    # =========================

    reason_max_abs = trial_max_abs > max_abs_thresh
    reason_ptp = trial_ptp > ptp_thresh
    reason_std_z = np.abs(robust_z_std) > std_z_thresh
    reason_flat = np.any(std < flat_std_thresh, axis=1)

    reject_mask = (
        reason_max_abs |
        reason_ptp |
        reason_std_z |
        reason_flat
    )

    keep_mask = ~reject_mask

    X_clean = X[keep_mask]
    y_clean = y[keep_mask]

    # =========================
    # SUMMARY
    # =========================

    if verbose:
        print("=" * 60)
        print("TRIAL REJECTION SUMMARY")
        print("=" * 60)

        print("Original trials :", len(X))
        print("Rejected trials :", int(np.sum(reject_mask)))
        print("Remaining trials:", len(X_clean))

        print("\nRejection reason counts:")
        print("Max abs exceeded :", int(np.sum(reason_max_abs)))
        print("PTP exceeded     :", int(np.sum(reason_ptp)))
        print("STD z-score bad  :", int(np.sum(reason_std_z)))
        print("Flat signal      :", int(np.sum(reason_flat)))

        print("\nRejected trial details:")
        rejected_indices = np.where(reject_mask)[0]

        if len(rejected_indices) == 0:
            print("None")
        else:
            for idx in rejected_indices:
                reasons = []

                if reason_max_abs[idx]:
                    reasons.append(
                        f"max_abs={trial_max_abs[idx]:.2f} > {max_abs_thresh}"
                    )

                if reason_ptp[idx]:
                    reasons.append(
                        f"ptp={trial_ptp[idx]:.2f} > {ptp_thresh}"
                    )

                if reason_std_z[idx]:
                    reasons.append(
                        f"std_z={robust_z_std[idx]:+.2f} > ±{std_z_thresh}"
                    )

                if reason_flat[idx]:
                    flat_channels = np.where(std[idx] < flat_std_thresh)[0]
                    reasons.append(
                        f"flat channels={flat_channels.tolist()}"
                    )

                print(f"Trial {idx}: " + " | ".join(reasons))

        print("\nClass balance before:")
        print(dict(zip(*np.unique(y, return_counts=True))))

        print("\nClass balance after:")
        print(dict(zip(*np.unique(y_clean, return_counts=True))))

    return {
        "X_clean": X_clean,
        "y_clean": y_clean,
        "keep_mask": keep_mask,
        "reject_mask": reject_mask,

        "reasons": {
            "max_abs": reason_max_abs,
            "ptp": reason_ptp,
            "std_z": reason_std_z,
            "flat": reason_flat,
        },

        "stats": {
            "trial_max_abs": trial_max_abs,
            "trial_ptp": trial_ptp,
            "trial_mean_std": trial_mean_std,
            "robust_z_std": robust_z_std,
        }
    }


result = reject_noisy_trials(
    X,
    y,
    max_abs_thresh=100,
    ptp_thresh=175,
    std_z_thresh=3.0
)

X = result["X_clean"]
y = result["y_clean"]

keep_mask = result["keep_mask"]
reject_mask = result["reject_mask"]
reasons = result["reasons"]
stats = result["stats"]

TRIAL REJECTION SUMMARY
Original trials : 6520
Rejected trials : 2230
Remaining trials: 4290

Rejection reason counts:
Max abs exceeded : 0
PTP exceeded     : 0
STD z-score bad  : 10
Flat signal      : 2220

Rejected trial details:
Trial 1: flat channels=[0]
Trial 2: flat channels=[0]
Trial 3: flat channels=[0]
Trial 12: flat channels=[0]
Trial 28: flat channels=[0]
Trial 35: flat channels=[2]
Trial 44: flat channels=[0]
Trial 75: flat channels=[2]
Trial 85: flat channels=[0]
Trial 89: flat channels=[0]
Trial 101: flat channels=[0]
Trial 105: flat channels=[0]
Trial 109: flat channels=[2]
Trial 121: flat channels=[0]
Trial 124: flat channels=[0]
Trial 130: flat channels=[0]
Trial 132: flat channels=[0]
Trial 134: flat channels=[0]
Trial 137: flat channels=[0]
Trial 141: flat channels=[0]
Trial 148: flat channels=[0]
Trial 157: flat channels=[1]
Trial 158: flat channels=[0]
Trial 159: flat channels=[0]
Trial 162: flat channels=[0]
Trial 164: flat channels=[0, 1]
Trial 168: flat channels

## Feature Extraction

In [62]:
import os
import json
import numpy as np
import pandas as pd

from scipy.signal import welch
from scipy.linalg import eigh

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
from torch.utils.data import TensorDataset, DataLoader

SAMPLING_RATE = 250
RANDOM_STATE = 42
BATCH_SIZE = 16

C3_IDX = 0
CZ_IDX = 1
C4_IDX = 2

N_CSP_COMPONENTS = 6
CSP_REG = 1e-6

BANDS = {
    "mu": (8, 12),
    "alpha": (8, 13),
    "low_alpha": (8, 10),
    "high_alpha": (10, 13),
    "beta": (13, 30),
}

FEATURE_SAVE_DIR = "FeatureModels"
FEATURE_SAVE_PATH = os.path.join(FEATURE_SAVE_DIR, "Features.npz")

os.makedirs(FEATURE_SAVE_DIR, exist_ok=True)

if X.ndim != 3:
    raise ValueError(f"Expected X shape (trials, channels, samples), got {X.shape}")

n_trials, n_channels, n_samples = X.shape

print("Original X shape:", X.shape)
print("Original y shape:", y.shape)
print("Channels:", n_channels)
print("Samples per trial:", n_samples)

def add_motor_imagery_virtual_channels(X, c3_idx, cz_idx, c4_idx):
    n_trials, n_channels, n_samples = X.shape

    X_parts = [X]
    channel_names = [f"ch{idx}" for idx in range(n_channels)]

    valid_c3 = c3_idx >= 0 and c3_idx < n_channels
    valid_cz = cz_idx >= 0 and cz_idx < n_channels
    valid_c4 = c4_idx >= 0 and c4_idx < n_channels

    if valid_c3:
        channel_names[c3_idx] = "C3"

    if valid_cz:
        channel_names[cz_idx] = "Cz"

    if valid_c4:
        channel_names[c4_idx] = "C4"

    if not (valid_c3 and valid_cz and valid_c4):
        print("Warning: C3/Cz/C4 not all valid. Returning original X only.")
        return X, channel_names

    c3 = X[:, c3_idx, :]
    cz = X[:, cz_idx, :]
    c4 = X[:, c4_idx, :]

    def add_channel(name, sig):
        X_parts.append(sig[:, None, :])
        channel_names.append(name)

    # ========================================================
    # 1. Clean / reference-subtracted channels
    # ========================================================

    c3_clean_025 = c3 - 0.25 * cz
    c3_clean_050 = c3 - 0.50 * cz
    c3_clean_075 = c3 - 0.75 * cz
    c3_clean_100 = c3 - 1.00 * cz

    c4_clean_025 = c4 - 0.25 * cz
    c4_clean_050 = c4 - 0.50 * cz
    c4_clean_075 = c4 - 0.75 * cz
    c4_clean_100 = c4 - 1.00 * cz

    add_channel("C3_clean_025Cz", c3_clean_025)
    add_channel("C3_clean_050Cz", c3_clean_050)
    add_channel("C3_clean_075Cz", c3_clean_075)
    add_channel("C3_clean_100Cz", c3_clean_100)

    add_channel("C4_clean_025Cz", c4_clean_025)
    add_channel("C4_clean_050Cz", c4_clean_050)
    add_channel("C4_clean_075Cz", c4_clean_075)
    add_channel("C4_clean_100Cz", c4_clean_100)
    add_channel("C3_clean", c3_clean_050)
    add_channel("C4_clean", c4_clean_050)

    # ========================================================
    # 2. Left-right contrast / asymmetry channels
    # ========================================================

    c3_minus_c4 = c3 - c4
    c4_minus_c3 = c4 - c3

    add_channel("C3_minus_C4", c3_minus_c4)
    add_channel("C4_minus_C3", c4_minus_c3)

    add_channel("C3clean025_minus_C4clean025", c3_clean_025 - c4_clean_025)
    add_channel("C3clean050_minus_C4clean050", c3_clean_050 - c4_clean_050)
    add_channel("C3clean075_minus_C4clean075", c3_clean_075 - c4_clean_075)
    add_channel("C3clean100_minus_C4clean100", c3_clean_100 - c4_clean_100)
    add_channel("C3clean_minus_C4clean", c3_clean_050 - c4_clean_050)

    # ========================================================
    # 3. Average/common-mode channels
    # ========================================================

    add_channel("C3_C4_mean", 0.5 * c3 + 0.5 * c4)
    add_channel("C3_Cz_mean", 0.5 * c3 + 0.5 * cz)
    add_channel("C4_Cz_mean", 0.5 * c4 + 0.5 * cz)
    add_channel("C3_Cz_C4_mean", (c3 + cz + c4) / 3.0)

    # ========================================================
    # 4. Smoothed spatial channels
    # ========================================================

    c3_smooth_421 = (4.0 / 7.0) * c3 + (2.0 / 7.0) * cz + (1.0 / 7.0) * c4
    c4_smooth_421 = (4.0 / 7.0) * c4 + (2.0 / 7.0) * cz + (1.0 / 7.0) * c3
    cz_smooth_211 = (2.0 / 4.0) * cz + (1.0 / 4.0) * c3 + (1.0 / 4.0) * c4

    add_channel("C3_smooth_421", c3_smooth_421)
    add_channel("Cz_smooth_211", cz_smooth_211)
    add_channel("C4_smooth_421", c4_smooth_421)

    # More aggressive local smoothing variants
    add_channel("C3_smooth_611", (6.0 / 8.0) * c3 + (1.0 / 8.0) * cz + (1.0 / 8.0) * c4)
    add_channel("C4_smooth_611", (6.0 / 8.0) * c4 + (1.0 / 8.0) * cz + (1.0 / 8.0) * c3)

    add_channel("C3_smooth_532", (5.0 / 10.0) * c3 + (3.0 / 10.0) * cz + (2.0 / 10.0) * c4)
    add_channel("C4_smooth_532", (5.0 / 10.0) * c4 + (3.0 / 10.0) * cz + (2.0 / 10.0) * c3)

    # ========================================================
    # 5. Laplacian-ish channels
    # ========================================================

    add_channel("C3_laplacian_simple", c3 - 0.5 * (cz + c4))
    add_channel("C4_laplacian_simple", c4 - 0.5 * (cz + c3))
    add_channel("Cz_laplacian_simple", cz - 0.5 * (c3 + c4))

    add_channel("C3_laplacian_strong", c3 - (0.75 * cz + 0.25 * c4))
    add_channel("C4_laplacian_strong", c4 - (0.75 * cz + 0.25 * c3))

    # ========================================================
    # 6. Ratio-like normalized asymmetry channels
    # ========================================================

    eps = 1e-8

    add_channel(
        "C3_minus_C4_over_abs_sum",
        (c3 - c4) / (np.abs(c3) + np.abs(c4) + eps)
    )

    add_channel(
        "C3clean_minus_C4clean_over_abs_sum",
        (c3_clean_050 - c4_clean_050) / (
            np.abs(c3_clean_050) + np.abs(c4_clean_050) + eps
        )
    )

    # ========================================================
    # 7. Cz-centered asymmetries
    # ========================================================

    add_channel("C3_minus_Cz", c3 - cz)
    add_channel("C4_minus_Cz", c4 - cz)
    add_channel("Cz_minus_C3", cz - c3)
    add_channel("Cz_minus_C4", cz - c4)

    add_channel("C3_minus_Cz_minus_C4_minus_Cz", (c3 - cz) - (c4 - cz))

    # ========================================================
    # 8. Weighted asymmetry blends
    # ========================================================

    add_channel("C3_dominant_asym_70_30", 0.7 * c3 - 0.3 * c4)
    add_channel("C4_dominant_asym_70_30", 0.7 * c4 - 0.3 * c3)

    add_channel("C3clean_dominant_asym_70_30", 0.7 * c3_clean_050 - 0.3 * c4_clean_050)
    add_channel("C4clean_dominant_asym_70_30", 0.7 * c4_clean_050 - 0.3 * c3_clean_050)

    X_aug = np.concatenate(X_parts, axis=1)

    return X_aug, channel_names

def safe_log(x, eps=1e-12):
    return np.log(np.maximum(x, eps))

def hjorth_mobility_complexity(sig):
    eps = 1e-12

    d1 = np.diff(sig)
    d2 = np.diff(d1)

    var0 = np.var(sig) + eps
    var1 = np.var(d1) + eps
    var2 = np.var(d2) + eps

    mobility = np.sqrt(var1 / var0)
    complexity = np.sqrt(var2 / var1) / (mobility + eps)

    return mobility, complexity


def bandpower(freqs, psd, fmin, fmax):
    idx = (freqs >= fmin) & (freqs <= fmax)

    if not np.any(idx):
        return 0.0

    return np.trapezoid(psd[idx], freqs[idx])


def clean_corr(sig_a, sig_b):
    corr = np.corrcoef(sig_a, sig_b)[0, 1]

    if np.isnan(corr) or np.isinf(corr):
        corr = 0.0

    return corr

def extract_curated_mi_features(X, channel_names, fs, bands):
    """
    Lean MI-focused feature set.

    Per channel:
    - log variance
    - Hjorth mobility
    - Hjorth complexity
    - relative bandpower for mu/alpha bands

    Selected pairwise:
    - C3 vs C4
    - C3_clean vs C4_clean
    - C3_minus_C4 vs C3clean_minus_C4clean
    - C3_clean vs C3_minus_C4
    - C4_clean vs C3_minus_C4

    This avoids feature explosion from all-to-all pairwise combinations.
    """

    n_trials, n_channels, n_samples = X.shape

    name_to_idx = {
        name: idx
        for idx, name in enumerate(channel_names)
    }

    useful_pair_names = [
        ("C3", "C4"),
        ("C3_clean", "C4_clean"),
        ("C3_minus_C4", "C3clean_minus_C4clean"),
        ("C3_clean", "C3_minus_C4"),
        ("C4_clean", "C3_minus_C4"),
    ]

    rows = []

    for trial_idx in range(n_trials):
        trial = X[trial_idx]
        row = {}

        for ch_idx in range(n_channels):
            sig = trial[ch_idx].astype(np.float64)
            ch_name = channel_names[ch_idx]

            sig_var = np.var(sig) + 1e-12

            mobility, complexity = hjorth_mobility_complexity(sig)

            row[f"{ch_name}_logvar"] = safe_log(sig_var)
            row[f"{ch_name}_hjorth_mobility"] = mobility
            row[f"{ch_name}_hjorth_complexity"] = complexity

            freqs, psd = welch(
                sig,
                fs=fs,
                nperseg=min(256, len(sig))
            )

            total_power = np.trapezoid(psd, freqs) + 1e-12

            for band_name, (fmin, fmax) in bands.items():
                bp = bandpower(freqs, psd, fmin, fmax)
                row[f"{ch_name}_{band_name}_relpower"] = bp / total_power

        for name_a, name_b in useful_pair_names:
            if name_a not in name_to_idx or name_b not in name_to_idx:
                continue

            idx_a = name_to_idx[name_a]
            idx_b = name_to_idx[name_b]

            sig_a = trial[idx_a].astype(np.float64)
            sig_b = trial[idx_b].astype(np.float64)

            var_a = np.var(sig_a) + 1e-12
            var_b = np.var(sig_b) + 1e-12

            row[f"{name_a}_minus_{name_b}_logvar_diff"] = safe_log(var_a) - safe_log(var_b)
            row[f"{name_a}_minus_{name_b}_corr"] = clean_corr(sig_a, sig_b)

        rows.append(row)

    features_df = pd.DataFrame(rows)
    features_df = features_df.replace([np.inf, -np.inf], np.nan)
    features_df = features_df.fillna(0.0)

    return features_df

indices = np.arange(len(y))

# 1. Train/val/test split
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y
)

idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp
)

X_train_base = X[idx_train]
X_val_base = X[idx_val]
X_test_base = X[idx_test]

print("\nBase split shapes before augmentation:")
print("X_train_base:", X_train_base.shape)
print("X_val_base  :", X_val_base.shape)
print("X_test_base :", X_test_base.shape)

print("\nBase class balance:")
print("Train:", np.unique(y_train, return_counts=True))
print("Val  :", np.unique(y_val, return_counts=True))
print("Test :", np.unique(y_test, return_counts=True))


# 2. Train-only augmentation to increase training set size and variability
rng = np.random.default_rng(RANDOM_STATE)

X_train_amp_up = X_train_base * 1.2
X_train_amp_down = X_train_base * 0.8
X_train_amp_up_1 = X_train_base * 1.4
X_train_amp_down_1 = X_train_base * 0.6

noise_std = 0.02 * np.std(X_train_base, axis=(0, 2), keepdims=True)

X_train_noise = X_train_base + rng.normal(
    loc=0.0,
    scale=noise_std,
    size=X_train_base.shape
)

X_train_aug_base = np.concatenate(
    [
        X_train_base
    ],
    axis=0
)

y_train = np.concatenate(
    [
        y_train
    ],
    axis=0
)

print("\nAfter train-only augmentation:")
print("X_train_aug_base:", X_train_aug_base.shape)
print("y_train         :", y_train.shape)

# 3. Add virtual channels to train/val/test
X_train_raw, channel_names_train = add_motor_imagery_virtual_channels(
    X_train_aug_base,
    C3_IDX,
    CZ_IDX,
    C4_IDX
)

X_val_raw, channel_names_val = add_motor_imagery_virtual_channels(
    X_val_base,
    C3_IDX,
    CZ_IDX,
    C4_IDX
)

X_test_raw, channel_names_test = add_motor_imagery_virtual_channels(
    X_test_base,
    C3_IDX,
    CZ_IDX,
    C4_IDX
)

if channel_names_train != channel_names_val or channel_names_train != channel_names_test:
    raise RuntimeError("Train/val/test channel names do not match after virtual channel generation.")

channel_names = channel_names_train

print("\nAfter virtual channels:")
print("X_train_raw:", X_train_raw.shape)
print("X_val_raw  :", X_val_raw.shape)
print("X_test_raw :", X_test_raw.shape)
print("Channel names:", channel_names)

# 4. Extract handcrafted features from train/val/test
X_train_hand_df = extract_curated_mi_features(
    X_train_raw,
    channel_names=channel_names,
    fs=SAMPLING_RATE,
    bands=BANDS
)

X_val_hand_df = extract_curated_mi_features(
    X_val_raw,
    channel_names=channel_names,
    fs=SAMPLING_RATE,
    bands=BANDS
)

X_test_hand_df = extract_curated_mi_features(
    X_test_raw,
    channel_names=channel_names,
    fs=SAMPLING_RATE,
    bands=BANDS
)

print("\nHandcrafted feature shapes:")
print("Hand train:", X_train_hand_df.shape)
print("Hand val  :", X_val_hand_df.shape)
print("Hand test :", X_test_hand_df.shape)

print("\nFinal split shapes:")
print("X_train_raw:", X_train_raw.shape)
print("X_val_raw  :", X_val_raw.shape)
print("X_test_raw :", X_test_raw.shape)

print("\nTrain class balance after augmentation:", np.unique(y_train, return_counts=True))
print("Val class balance                    :", np.unique(y_val, return_counts=True))
print("Test class balance                   :", np.unique(y_test, return_counts=True))

class SimpleCSP:
    def __init__(self, n_components=6, reg=1e-6):
        self.n_components = n_components
        self.reg = reg
        self.filters_ = None

    def _covariance(self, trial):
        cov = trial @ trial.T
        cov = cov / (np.trace(cov) + 1e-12)
        return cov

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y)

        classes = np.unique(y)

        if len(classes) != 2:
            raise ValueError(f"CSP needs exactly 2 classes, got {classes}")

        n_trials, n_channels, n_samples = X.shape

        covs = []

        for cls in classes:
            X_cls = X[y == cls]

            cls_covs = []

            for trial in X_cls:
                cls_covs.append(self._covariance(trial))

            covs.append(np.mean(cls_covs, axis=0))

        cov_0 = covs[0]
        cov_1 = covs[1]

        composite_cov = cov_0 + cov_1

        cov_0 = cov_0 + self.reg * np.eye(n_channels)
        composite_cov = composite_cov + self.reg * np.eye(n_channels)

        eigenvalues, eigenvectors = eigh(cov_0, composite_cov)

        sorted_indices = np.argsort(eigenvalues)

        n_components = min(self.n_components, n_channels)

        half = n_components // 2

        if n_components % 2 == 0:
            selected_indices = np.concatenate([
                sorted_indices[:half],
                sorted_indices[-half:]
            ])
        else:
            selected_indices = np.concatenate([
                sorted_indices[:half + 1],
                sorted_indices[-half:]
            ])

        self.filters_ = eigenvectors[:, selected_indices].T

        return self

    def transform(self, X):
        if self.filters_ is None:
            raise RuntimeError("CSP must be fitted before transform.")

        X = np.asarray(X, dtype=np.float64)

        out = []

        for trial in X:
            projected = self.filters_ @ trial

            var = np.var(projected, axis=1)
            var_norm = var / (np.sum(var) + 1e-12)

            features = np.log(var_norm + 1e-12)
            out.append(features)

        return np.asarray(out)

# 5. Fit CSP on training data and transform train/val/test
csp = SimpleCSP(
    n_components=N_CSP_COMPONENTS,
    reg=CSP_REG
)

csp.fit(X_train_raw, y_train)

X_train_csp = csp.transform(X_train_raw)
X_val_csp = csp.transform(X_val_raw)
X_test_csp = csp.transform(X_test_raw)

csp_feature_names = [f"csp_{i}" for i in range(X_train_csp.shape[1])]

X_train_csp_df = pd.DataFrame(X_train_csp, columns=csp_feature_names)
X_val_csp_df = pd.DataFrame(X_val_csp, columns=csp_feature_names)
X_test_csp_df = pd.DataFrame(X_test_csp, columns=csp_feature_names)

print("\nCSP feature shapes:")
print("Train CSP:", X_train_csp_df.shape)
print("Val CSP  :", X_val_csp_df.shape)
print("Test CSP :", X_test_csp_df.shape)

X_train_features_df = pd.concat(
    [X_train_hand_df, X_train_csp_df],
    axis=1
)

X_val_features_df = pd.concat(
    [X_val_hand_df, X_val_csp_df],
    axis=1
)

X_test_features_df = pd.concat(
    [X_test_hand_df, X_test_csp_df],
    axis=1
)

feature_names = X_train_features_df.columns.tolist()

print("\nFinal unscaled feature shapes:")
print("Train:", X_train_features_df.shape)
print("Val  :", X_val_features_df.shape)
print("Test :", X_test_features_df.shape)

print("\nTotal final features:", len(feature_names))
print("First 20 feature names:")
print(feature_names[:20])

# 6. Data Prep and saving
feature_scaler = StandardScaler()

X_train_features = feature_scaler.fit_transform(X_train_features_df)
X_val_features = feature_scaler.transform(X_val_features_df)
X_test_features = feature_scaler.transform(X_test_features_df)

print("\nScaled feature arrays:")
print("X_train_features:", X_train_features.shape)
print("X_val_features  :", X_val_features.shape)
print("X_test_features :", X_test_features.shape)

print("\nOriginal label values:")
print("Train:", np.unique(y_train))
print("Val  :", np.unique(y_val))
print("Test :", np.unique(y_test))

if np.min(y_train) == 1:
    y_train_fixed = y_train - 1
    y_val_fixed = y_val - 1
    y_test_fixed = y_test - 1
else:
    y_train_fixed = y_train.copy()
    y_val_fixed = y_val.copy()
    y_test_fixed = y_test.copy()

print("\nFixed label values:")
print("Train:", np.unique(y_train_fixed))
print("Val  :", np.unique(y_val_fixed))
print("Test :", np.unique(y_test_fixed))

def split_features_by_class(X_features, y_labels):
    out = {}

    classes = np.unique(y_labels)

    for cls in classes:
        out[int(cls)] = X_features[y_labels == cls]

    return out


X_train_unscaled = X_train_features_df.values.astype(np.float64)
X_val_unscaled = X_val_features_df.values.astype(np.float64)
X_test_unscaled = X_test_features_df.values.astype(np.float64)

train_unscaled_by_class = split_features_by_class(X_train_unscaled, y_train_fixed)
val_unscaled_by_class = split_features_by_class(X_val_unscaled, y_val_fixed)
test_unscaled_by_class = split_features_by_class(X_test_unscaled, y_test_fixed)

train_scaled_by_class = split_features_by_class(X_train_features, y_train_fixed)
val_scaled_by_class = split_features_by_class(X_val_features, y_val_fixed)
test_scaled_by_class = split_features_by_class(X_test_features, y_test_fixed)

metadata = {
    "description": "Lean curated handcrafted + CSP EEG features for MI class separability analysis.",
    "sampling_rate": SAMPLING_RATE,
    "random_state": RANDOM_STATE,
    "n_csp_components": N_CSP_COMPONENTS,
    "csp_reg": CSP_REG,
    "bands": BANDS,
    "feature_count": len(feature_names),
    "channel_names": channel_names,
    "feature_names": feature_names,
    "train_shape_unscaled": X_train_unscaled.shape,
    "val_shape_unscaled": X_val_unscaled.shape,
    "test_shape_unscaled": X_test_unscaled.shape,
    "train_shape_scaled": X_train_features.shape,
    "val_shape_scaled": X_val_features.shape,
    "test_shape_scaled": X_test_features.shape,
    "classes": np.unique(y_train_fixed).tolist()
}

np.savez_compressed(
    FEATURE_SAVE_PATH,

    metadata_json=np.array(json.dumps(metadata), dtype=object),
    feature_names=np.array(feature_names, dtype=object),
    channel_names=np.array(channel_names, dtype=object),

    idx_train=idx_train,
    idx_val=idx_val,
    idx_test=idx_test,

    y_train=y_train_fixed,
    y_val=y_val_fixed,
    y_test=y_test_fixed,

    X_train_unscaled=X_train_unscaled,
    X_val_unscaled=X_val_unscaled,
    X_test_unscaled=X_test_unscaled,

    X_train_scaled=X_train_features,
    X_val_scaled=X_val_features,
    X_test_scaled=X_test_features,

    X_train_unscaled_class_0=train_unscaled_by_class.get(0, np.empty((0, X_train_unscaled.shape[1]))),
    X_train_unscaled_class_1=train_unscaled_by_class.get(1, np.empty((0, X_train_unscaled.shape[1]))),

    X_val_unscaled_class_0=val_unscaled_by_class.get(0, np.empty((0, X_val_unscaled.shape[1]))),
    X_val_unscaled_class_1=val_unscaled_by_class.get(1, np.empty((0, X_val_unscaled.shape[1]))),

    X_test_unscaled_class_0=test_unscaled_by_class.get(0, np.empty((0, X_test_unscaled.shape[1]))),
    X_test_unscaled_class_1=test_unscaled_by_class.get(1, np.empty((0, X_test_unscaled.shape[1]))),

    X_train_scaled_class_0=train_scaled_by_class.get(0, np.empty((0, X_train_features.shape[1]))),
    X_train_scaled_class_1=train_scaled_by_class.get(1, np.empty((0, X_train_features.shape[1]))),

    X_val_scaled_class_0=val_scaled_by_class.get(0, np.empty((0, X_val_features.shape[1]))),
    X_val_scaled_class_1=val_scaled_by_class.get(1, np.empty((0, X_val_features.shape[1]))),

    X_test_scaled_class_0=test_scaled_by_class.get(0, np.empty((0, X_test_features.shape[1]))),
    X_test_scaled_class_1=test_scaled_by_class.get(1, np.empty((0, X_test_features.shape[1])))
)

print("\nSaved feature separability file:")
print(FEATURE_SAVE_PATH)

print("\nClass-separated scaled train:")
print("Class 0:", train_scaled_by_class.get(0, np.empty((0, X_train_features.shape[1]))).shape)
print("Class 1:", train_scaled_by_class.get(1, np.empty((0, X_train_features.shape[1]))).shape)

Original X shape: (6475, 3, 1751)
Original y shape: (6475,)
Channels: 3
Samples per trial: 1751

Base split shapes before augmentation:
X_train_base: (4532, 3, 1751)
X_val_base  : (971, 3, 1751)
X_test_base : (972, 3, 1751)

Base class balance:
Train: (array([1, 2]), array([2264, 2268]))
Val  : (array([1, 2]), array([485, 486]))
Test : (array([1, 2]), array([486, 486]))

After train-only augmentation:
X_train_aug_base: (4532, 3, 1751)
y_train         : (4532,)

After virtual channels:
X_train_raw: (4532, 47, 1751)
X_val_raw  : (971, 47, 1751)
X_test_raw : (972, 47, 1751)
Channel names: ['C3', 'Cz', 'C4', 'C3_clean_025Cz', 'C3_clean_050Cz', 'C3_clean_075Cz', 'C3_clean_100Cz', 'C4_clean_025Cz', 'C4_clean_050Cz', 'C4_clean_075Cz', 'C4_clean_100Cz', 'C3_clean', 'C4_clean', 'C3_minus_C4', 'C4_minus_C3', 'C3clean025_minus_C4clean025', 'C3clean050_minus_C4clean050', 'C3clean075_minus_C4clean075', 'C3clean100_minus_C4clean100', 'C3clean_minus_C4clean', 'C3_C4_mean', 'C3_Cz_mean', 'C4_Cz_mean',

## Data Prep

In [90]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler

BATCH_SIZE = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

X_train_seq = np.transpose(X_train_raw, (0, 2, 1))
X_val_seq = np.transpose(X_val_raw, (0, 2, 1))
X_test_seq = np.transpose(X_test_raw, (0, 2, 1))

n_train, n_samples, n_channels = X_train_seq.shape

print("Raw sequence shapes before scaling:")
print("X_train_seq:", X_train_seq.shape)
print("X_val_seq  :", X_val_seq.shape)
print("X_test_seq :", X_test_seq.shape)

# Scale raw EEG using train only
seq_scaler = StandardScaler()

X_train_seq_scaled = seq_scaler.fit_transform(
    X_train_seq.reshape(-1, n_channels)
).reshape(X_train_seq.shape)

X_val_seq_scaled = seq_scaler.transform(
    X_val_seq.reshape(-1, n_channels)
).reshape(X_val_seq.shape)

X_test_seq_scaled = seq_scaler.transform(
    X_test_seq.reshape(-1, n_channels)
).reshape(X_test_seq.shape)

print("\nFeature shapes:")
print("X_train_features:", X_train_features.shape)
print("X_val_features  :", X_val_features.shape)
print("X_test_features :", X_test_features.shape)

X_train_seq_t = torch.tensor(X_train_seq_scaled, dtype=torch.float32).to(device)
X_val_seq_t = torch.tensor(X_val_seq_scaled, dtype=torch.float32).to(device)
X_test_seq_t = torch.tensor(X_test_seq_scaled, dtype=torch.float32).to(device)

X_train_feat_t = torch.tensor(X_train_features, dtype=torch.float32).to(device)
X_val_feat_t = torch.tensor(X_val_features, dtype=torch.float32).to(device)
X_test_feat_t = torch.tensor(X_test_features, dtype=torch.float32).to(device)

y_train_t = torch.tensor(y_train_fixed, dtype=torch.long).to(device)
y_val_t = torch.tensor(y_val_fixed, dtype=torch.long).to(device)
y_test_t = torch.tensor(y_test_fixed, dtype=torch.long).to(device)

train_dataset = TensorDataset(
    X_train_seq_t,
    X_train_feat_t,
    y_train_t
)

val_dataset = TensorDataset(
    X_val_seq_t,
    X_val_feat_t,
    y_val_t
)

test_dataset = TensorDataset(
    X_test_seq_t,
    X_test_feat_t,
    y_test_t
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

seq_input_size = X_train_seq_t.shape[2]
feature_input_size = X_train_feat_t.shape[1]
num_classes = len(torch.unique(y_train_t))

print("\nReady for fused model:")
print("seq_input_size    :", seq_input_size)
print("feature_input_size:", feature_input_size)
print("num_classes       :", num_classes)

Using device: cuda
Raw sequence shapes before scaling:
X_train_seq: (370, 1175, 47)
X_val_seq  : (79, 1175, 47)
X_test_seq : (80, 1175, 47)

Feature shapes:
X_train_features: (370, 392)
X_val_features  : (79, 392)
X_test_features : (80, 392)

Ready for fused model:
seq_input_size    : 47
feature_input_size: 392
num_classes       : 2


## LSTM Architecture

In [91]:
import torch
import torch.nn as nn
from torchinfo import summary


class BraindanceCNNLSTMFusion(nn.Module):

    def __init__(
        self,
        seq_input_size,
        feature_input_size,
        num_classes=2,
        bidirectional=False
    ):
        super().__init__()

        self.bidirectional = bidirectional
        self.cnn = nn.Sequential(

            nn.Conv1d(
                in_channels=seq_input_size,
                out_channels=128,
                kernel_size=15,
                padding=7
            ),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.MaxPool1d(kernel_size=4),

            nn.Conv1d(
                in_channels=128,
                out_channels=256,
                kernel_size=9,
                padding=4
            ),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.MaxPool1d(kernel_size=4),
        )

        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            dropout=0.45,
            bidirectional=bidirectional
        )

        lstm_output_size = 128 if bidirectional else 64

        self.lstm_projector = nn.Sequential(
            nn.Linear(lstm_output_size, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(32, 8),
            nn.BatchNorm1d(8),
            nn.ReLU(),
            nn.Dropout(0.25)
        )

        self.feature_mlp = nn.Sequential(

            nn.Linear(feature_input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(128, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.25)
        )

        self.classifier = nn.Sequential(

            nn.Linear(16+8, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(0.25),

            nn.Linear(16, 4),
            nn.BatchNorm1d(4),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(4, num_classes)
        )

        self.gate = nn.Linear(24, 2)

    def forward(self, x_seq, x_features):

        # (B,T,C) -> (B,C,T)
        x_seq = x_seq.transpose(1, 2)

        cnn_out = self.cnn(x_seq)

        # (B,C,T) -> (B,T,C)
        cnn_out = cnn_out.transpose(1, 2)

        _, (h_n, _) = self.lstm(cnn_out)

        if self.bidirectional:
            lstm_final = torch.cat(
                [h_n[-2], h_n[-1]],
                dim=1
            )
        else:
            lstm_final = h_n[-1]

        lstm_features = self.lstm_projector(lstm_final)
        mlp_features = self.feature_mlp(x_features)

        combined = torch.cat(
            [
                lstm_features,
                mlp_features
            ],
            dim=1
        )

        gate_logits = self.gate(combined)
        weights = torch.softmax(gate_logits, dim=1)

        lstm_features = (
            lstm_features *
            weights[:, 0:1]
        )

        mlp_features = (
            mlp_features *
            weights[:, 1:2]
        )

        fused = torch.cat(
            [
                lstm_features,
                mlp_features
            ],
            dim=1
        )

        logits = self.classifier(fused)

        self.last_lstm_weight = weights[:, 0].mean().detach()
        self.last_mlp_weight = weights[:, 1].mean().detach()

        return logits
    

model = BraindanceCNNLSTMFusion(
    seq_input_size=seq_input_size,
    feature_input_size=feature_input_size,
    num_classes=num_classes,
    bidirectional=True
).to(device)

summary(
    model,
    input_data=(
        torch.randn(16, 1175, seq_input_size).to(device),
        torch.randn(16, feature_input_size).to(device)
    ),
    depth=5,
    col_names=[
        "input_size",
        "output_size",
        "num_params",
        "trainable"
    ]
)

c:\Stored Data\BrainDance\venv\Lib\site-packages\torch\nn\modules\rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.45 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Trainable
BraindanceCNNLSTMFusion                  [16, 1175, 47]            [16, 2]                   --                        True
├─Sequential: 1-1                        [16, 47, 1175]            [16, 256, 73]             --                        True
│    └─Conv1d: 2-1                       [16, 47, 1175]            [16, 128, 1175]           90,368                    True
│    └─BatchNorm1d: 2-2                  [16, 128, 1175]           [16, 128, 1175]           256                       True
│    └─ReLU: 2-3                         [16, 128, 1175]           [16, 128, 1175]           --                        --
│    └─Dropout: 2-4                      [16, 128, 1175]           [16, 128, 1175]           --                        --
│    └─MaxPool1d: 2-5                    [16, 128, 1175]           [16, 128, 293]            --                        --
│    └─Co

## Training Loop

In [92]:
import copy
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

EPOCHS = 150
PATIENCE = 8

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0001

SAVE_LOWEST_VAL_LOSS_PATH = "best_fused_lowest_val_loss.pt"
SAVE_HIGHEST_VAL_AUC_PATH = "best_fused_highest_val_auc.pt"
SAVE_HIGHEST_VAL_ACC_PATH = "best_fused_highest_val_acc.pt"
SAVE_HIGHEST_VAL_ULTA_RATIO_PATH = "best_fused_highest_val_ulta_ratio.pt"

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3,
    cooldown=0,
    min_lr=1e-6
)

history = {
    "train_loss": [],
    "val_loss": [],

    "train_acc": [],
    "val_acc": [],

    "train_bal_acc": [],
    "val_bal_acc": [],

    "train_auc": [],
    "val_auc": [],

    "train_f1": [],
    "val_f1": [],

    "train_ulta_ratio": [],
    "val_ulta_ratio": []
}

best_val_loss = float("inf")
best_val_auc = -float("inf")
best_val_acc = -float("inf")
best_val_ulta_ratio = -float("inf")

best_val_loss_state = None
best_val_auc_state = None
best_val_acc_state = None
best_val_ulta_ratio_state = None

best_val_loss_epoch = None
best_val_auc_epoch = None
best_val_acc_epoch = None
best_val_ulta_ratio_epoch = None

epochs_without_improvement = 0


def safe_auc_score(labels, probs):
    try:
        return roc_auc_score(labels, probs)
    except ValueError:
        return np.nan


def compute_ulta_ratio(auc, acc, loss):
    if np.isnan(auc):
        return np.nan

    return auc * acc * (1.0 / (1.0 + loss))


def evaluate_fused_model(model, data_loader, criterion, device):
    model.eval()

    total_loss = 0.0

    all_labels = []
    all_preds = []
    all_probs_class_1 = []

    with torch.no_grad():
        for X_seq_batch, X_feat_batch, y_batch in data_loader:
            X_seq_batch = X_seq_batch.to(device)
            X_feat_batch = X_feat_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_seq_batch, X_feat_batch)
            loss = criterion(logits, y_batch)

            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            batch_size = y_batch.size(0)
            total_loss += loss.item() * batch_size

            all_labels.extend(y_batch.detach().cpu().numpy())
            all_preds.extend(preds.detach().cpu().numpy())

            if probs.shape[1] == 2:
                all_probs_class_1.extend(probs[:, 1].detach().cpu().numpy())
            else:
                all_probs_class_1.extend([np.nan] * batch_size)

    avg_loss = total_loss / len(data_loader.dataset)

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs_class_1 = np.array(all_probs_class_1)

    acc = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, zero_division=0)
    auc = safe_auc_score(all_labels, all_probs_class_1)

    ulta_ratio = compute_ulta_ratio(
        auc=auc,
        acc=acc,
        loss=avg_loss
    )

    return {
        "loss": avg_loss,
        "acc": acc,
        "bal_acc": bal_acc,
        "auc": auc,
        "f1": f1,
        "ulta_ratio": ulta_ratio,
        "labels": all_labels,
        "preds": all_preds,
        "probs_class_1": all_probs_class_1
    }


for epoch in range(1, EPOCHS + 1):
    model.train()

    running_train_loss = 0.0

    train_labels = []
    train_preds = []
    train_probs_class_1 = []

    for X_seq_batch, X_feat_batch, y_batch in train_loader:
        X_seq_batch = X_seq_batch.to(device)
        X_feat_batch = X_feat_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_seq_batch, X_feat_batch)
        loss = criterion(logits, y_batch)

        loss.backward()

        # Useful for LSTM stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        probs = torch.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)

        batch_size = y_batch.size(0)
        running_train_loss += loss.item() * batch_size

        train_labels.extend(y_batch.detach().cpu().numpy())
        train_preds.extend(preds.detach().cpu().numpy())

        if probs.shape[1] == 2:
            train_probs_class_1.extend(probs[:, 1].detach().cpu().numpy())
        else:
            train_probs_class_1.extend([np.nan] * batch_size)

    train_loss = running_train_loss / len(train_loader.dataset)

    train_labels = np.array(train_labels)
    train_preds = np.array(train_preds)
    train_probs_class_1 = np.array(train_probs_class_1)

    train_acc = accuracy_score(train_labels, train_preds)
    train_bal_acc = balanced_accuracy_score(train_labels, train_preds)
    train_f1 = f1_score(train_labels, train_preds, zero_division=0)
    train_auc = safe_auc_score(train_labels, train_probs_class_1)

    train_ulta_ratio = compute_ulta_ratio(
        auc=train_auc,
        acc=train_acc,
        loss=train_loss
    )

    val_results = evaluate_fused_model(
        model=model,
        data_loader=val_loader,
        criterion=criterion,
        device=device
    )

    val_loss = val_results["loss"]
    val_acc = val_results["acc"]
    val_bal_acc = val_results["bal_acc"]
    val_auc = val_results["auc"]
    val_f1 = val_results["f1"]
    val_ulta_ratio = val_results["ulta_ratio"]

    scheduler.step(val_auc)
    current_lr = optimizer.param_groups[0]["lr"]

    # -------------------------
    # Save history
    # -------------------------

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    history["train_bal_acc"].append(train_bal_acc)
    history["val_bal_acc"].append(val_bal_acc)

    history["train_auc"].append(train_auc)
    history["val_auc"].append(val_auc)

    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    history["train_ulta_ratio"].append(train_ulta_ratio)
    history["val_ulta_ratio"].append(val_ulta_ratio)

    improved = False

    # -------------------------
    # Save lowest val loss
    # -------------------------

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_loss_state = copy.deepcopy(model.state_dict())
        best_val_loss_epoch = epoch

        # torch.save(best_val_loss_state, SAVE_LOWEST_VAL_LOSS_PATH)

        improved = True

    # -------------------------
    # Save highest val AUC
    # -------------------------

    if not np.isnan(val_auc) and val_auc > best_val_auc:
        best_val_auc = val_auc
        best_val_auc_state = copy.deepcopy(model.state_dict())
        best_val_auc_epoch = epoch

        # torch.save(best_val_auc_state, SAVE_HIGHEST_VAL_AUC_PATH)

        improved = True

    # -------------------------
    # Save highest val accuracy
    # -------------------------

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_acc_state = copy.deepcopy(model.state_dict())
        best_val_acc_epoch = epoch

        # torch.save(best_val_acc_state, SAVE_HIGHEST_VAL_ACC_PATH)

        improved = True

    # -------------------------
    # Save highest val ulta ratio
    # -------------------------

    if not np.isnan(val_ulta_ratio) and val_ulta_ratio > best_val_ulta_ratio:
        best_val_ulta_ratio = val_ulta_ratio
        best_val_ulta_ratio_state = copy.deepcopy(model.state_dict())
        best_val_ulta_ratio_epoch = epoch

        # torch.save(best_val_ulta_ratio_state, SAVE_HIGHEST_VAL_ULTA_RATIO_PATH)

        improved = True

    if improved:
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
            f"Epoch [{epoch:03d}/{EPOCHS}] "
            f"LR: {current_lr:.2e} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Val Bal Acc: {val_bal_acc:.4f} | "
            f"Val AUC: {val_auc:.4f} | "
            f"Val F1: {val_f1:.4f} | "
            f"Val Ulta: {val_ulta_ratio:.4f}"
            f"LSTM Gate: {model.last_lstm_weight:.3f} | "
            f"MLP Gate: {model.last_mlp_weight:.3f} | "
        )
    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping triggered after {epoch} epochs.")
        break


# ============================================================
# TRAINING COMPLETE
# ============================================================

print("\n================ TRAINING COMPLETE ================")
print(f"Best Lowest Val Loss : {best_val_loss:.4f} at epoch {best_val_loss_epoch}")
print(f"Best Highest Val AUC  : {best_val_auc:.4f} at epoch {best_val_auc_epoch}")
print(f"Best Highest Val Acc  : {best_val_acc:.4f} at epoch {best_val_acc_epoch}")
print(f"Best Highest Ulta     : {best_val_ulta_ratio:.4f} at epoch {best_val_ulta_ratio_epoch}")

print("\nSaved models:")
print(SAVE_LOWEST_VAL_LOSS_PATH)
print(SAVE_HIGHEST_VAL_AUC_PATH)
print(SAVE_HIGHEST_VAL_ACC_PATH)
print(SAVE_HIGHEST_VAL_ULTA_RATIO_PATH)


# ============================================================
# TEST BEST MODELS
# ============================================================

def test_saved_fused_state(model, state_dict, test_loader, criterion, device, model_name):
    if state_dict is None:
        raise ValueError(f"No saved state found for {model_name}.")

    model.load_state_dict(state_dict)
    model.to(device)

    results = evaluate_fused_model(
        model=model,
        data_loader=test_loader,
        criterion=criterion,
        device=device
    )

    return {
        "name": model_name,
        "loss": results["loss"],
        "acc": results["acc"],
        "bal_acc": results["bal_acc"],
        "auc": results["auc"],
        "f1": results["f1"],
        "ulta_ratio": results["ulta_ratio"],
        "labels": results["labels"],
        "preds": results["preds"],
        "probs_class_1": results["probs_class_1"]
    }


test_results = []

test_results.append(
    test_saved_fused_state(
        model=model,
        state_dict=best_val_loss_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Lowest Val Loss"
    )
)

test_results.append(
    test_saved_fused_state(
        model=model,
        state_dict=best_val_auc_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Highest Val AUC"
    )
)

test_results.append(
    test_saved_fused_state(
        model=model,
        state_dict=best_val_acc_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Highest Val Acc"
    )
)

test_results.append(
    test_saved_fused_state(
        model=model,
        state_dict=best_val_ulta_ratio_state,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        model_name="Highest Val Ulta"
    )
)

print("\n================ TEST COMPARISON ================")

print(
    f"{'Model':<24} "
    f"{'Test Loss':<12} "
    f"{'Test Acc':<10} "
    f"{'Test Bal Acc':<14} "
    f"{'Test AUC':<10} "
    f"{'Test F1':<10} "
    f"{'Test Ulta':<12}"
)

print("-" * 100)

for result in test_results:
    print(
        f"{result['name']:<24} "
        f"{result['loss']:<12.4f} "
        f"{result['acc']:<10.4f} "
        f"{result['bal_acc']:<14.4f} "
        f"{result['auc']:<10.4f} "
        f"{result['f1']:<10.4f} "
        f"{result['ulta_ratio']:<12.4f}"
    )


for result in test_results:
    print(f"\n================ {result['name']} TEST REPORT ================")

    print(
        classification_report(
            result["labels"],
            result["preds"],
            zero_division=0
        )
    )

epochs_ran = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_loss"], label="Train Loss")
plt.plot(epochs_ran, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fused Model: Train Loss vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_acc"], label="Train Accuracy")
plt.plot(epochs_ran, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Fused Model: Train Accuracy vs Validation Accuracy")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["val_auc"], label="Validation AUC")
plt.xlabel("Epoch")
plt.ylabel("AUC")
plt.title("Fused Model: Validation AUC")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_f1"], label="Train F1")
plt.plot(epochs_ran, history["val_f1"], label="Validation F1")
plt.xlabel("Epoch")
plt.ylabel("F1 Score")
plt.title("Fused Model: Train F1 vs Validation F1")
plt.legend()
plt.grid(True)
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(epochs_ran, history["train_ulta_ratio"], label="Train Ulta Ratio")
plt.plot(epochs_ran, history["val_ulta_ratio"], label="Validation Ulta Ratio")
plt.xlabel("Epoch")
plt.ylabel("Ulta Ratio")
plt.title("Fused Model: Train vs Validation Ulta Ratio")
plt.legend()
plt.grid(True)
plt.show()


for result in test_results:
    cm = confusion_matrix(result["labels"], result["preds"])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Class 0", "Class 1"]
    )

    disp.plot(values_format="d")
    plt.title(f"Test Confusion Matrix - {result['name']}")
    plt.grid(False)
    plt.show()

Epoch [001/150] LR: 1.00e-03 | Train Loss: 0.7442 | Val Loss: 0.6933 | Train Acc: 0.5405 | Val Acc: 0.5063 | Val Bal Acc: 0.5000 | Val AUC: 0.5885 | Val F1: 0.0000 | Val Ulta: 0.1760LSTM Gate: 0.429 | MLP Gate: 0.571 | 
Epoch [002/150] LR: 1.00e-03 | Train Loss: 0.7189 | Val Loss: 0.6874 | Train Acc: 0.5432 | Val Acc: 0.5696 | Val Bal Acc: 0.5651 | Val AUC: 0.5462 | Val F1: 0.3200 | Val Ulta: 0.1844LSTM Gate: 0.399 | MLP Gate: 0.601 | 
Epoch [003/150] LR: 1.00e-03 | Train Loss: 0.7104 | Val Loss: 0.6940 | Train Acc: 0.5270 | Val Acc: 0.4937 | Val Bal Acc: 0.4901 | Val AUC: 0.4769 | Val F1: 0.2857 | Val Ulta: 0.1390LSTM Gate: 0.384 | MLP Gate: 0.616 | 
Epoch [004/150] LR: 1.00e-03 | Train Loss: 0.6911 | Val Loss: 0.6943 | Train Acc: 0.5595 | Val Acc: 0.5063 | Val Bal Acc: 0.5032 | Val AUC: 0.4859 | Val F1: 0.3390 | Val Ulta: 0.1452LSTM Gate: 0.387 | MLP Gate: 0.613 | 
Epoch [005/150] LR: 5.00e-04 | Train Loss: 0.6968 | Val Loss: 0.6921 | Train Acc: 0.5514 | Val Acc: 0.5063 | Val Bal Acc

KeyboardInterrupt: 

## Saving

In [ ]:
import os
import copy
import joblib
import numpy as np
import torch


LIVE_CHECKPOINT_NAME = "highest_val_auc"   # lowest_val_loss, highest_val_auc, highest_val_acc, highest_val_ulta_ratio


CHECKPOINT_STATES = {
    "lowest_val_loss": {
        "state": best_val_loss_state,
        "best_metric": best_val_loss,
        "best_epoch": best_val_loss_epoch,
        "metric_name": "val_loss",
        "higher_is_better": False,
    },
    "highest_val_auc": {
        "state": best_val_auc_state,
        "best_metric": best_val_auc,
        "best_epoch": best_val_auc_epoch,
        "metric_name": "val_auc",
        "higher_is_better": True,
    },
    "highest_val_acc": {
        "state": best_val_acc_state,
        "best_metric": best_val_acc,
        "best_epoch": best_val_acc_epoch,
        "metric_name": "val_acc",
        "higher_is_better": True,
    },
    "highest_val_ulta_ratio": {
        "state": best_val_ulta_ratio_state,
        "best_metric": best_val_ulta_ratio,
        "best_epoch": best_val_ulta_ratio_epoch,
        "metric_name": "val_ulta_ratio",
        "higher_is_better": True,
    },
}

if LIVE_CHECKPOINT_NAME not in CHECKPOINT_STATES:
    raise ValueError(
        f"Unknown checkpoint name: {LIVE_CHECKPOINT_NAME}"
    )

LIVE_STATE = CHECKPOINT_STATES[LIVE_CHECKPOINT_NAME]["state"]

if LIVE_STATE is None:
    raise RuntimeError(
        f"{LIVE_CHECKPOINT_NAME} state is None."
    )


# ============================================================
# REBUILD MODEL
# ============================================================

live_model = BraindanceCNNLSTMFusion(
    seq_input_size=seq_input_size,
    feature_input_size=feature_input_size,
    num_classes=num_classes,
    bidirectional=True
).to(device)

live_model.load_state_dict(LIVE_STATE)
live_model.eval()


# ============================================================
# VERIFY CHECKPOINT LOADS CORRECTLY
# ============================================================

test_model = BraindanceCNNLSTMFusion(
    seq_input_size=seq_input_size,
    feature_input_size=feature_input_size,
    num_classes=num_classes,
    bidirectional=True
).to(device)

test_model.load_state_dict(copy.deepcopy(LIVE_STATE))

print("Checkpoint verification successful.")


# ============================================================
# SAVE LOCATION
# ============================================================

MODEL_SAVE_DIR = "Model"
MODEL_SAVE_PATH = os.path.join(
    MODEL_SAVE_DIR,
    "braindance.joblib"
)

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


def safe_get(name, default=None):
    return globals()[name] if name in globals() else default


# ============================================================
# FEATURE PIPELINE OBJECTS
# ============================================================

saved_feature_scaler = safe_get("feature_scaler")
saved_seq_scaler = safe_get("seq_scaler")
saved_csp = safe_get("csp")

saved_channel_names = safe_get("channel_names")
saved_feature_names = safe_get("feature_names")

saved_bands = safe_get("BANDS")

saved_c3_idx = safe_get("C3_IDX")
saved_cz_idx = safe_get("CZ_IDX")
saved_c4_idx = safe_get("C4_IDX")

saved_sampling_rate = safe_get(
    "SAMPLING_RATE",
    safe_get("FS")
)

saved_n_csp_components = safe_get("N_CSP_COMPONENTS")
saved_csp_reg = safe_get("CSP_REG")


# ============================================================
# SHAPES
# ============================================================

saved_seq_input_size = int(seq_input_size)
saved_feature_input_size = int(feature_input_size)
saved_num_classes = int(num_classes)


# ============================================================
# CHECKPOINT
# ============================================================

checkpoint = {

    "checkpoint_name":
        LIVE_CHECKPOINT_NAME,

    "model_type":
        "cnn_lstm_feature_fusion",

    "model_class_name":
        live_model.__class__.__name__,

    "best_metric_name":
        CHECKPOINT_STATES[LIVE_CHECKPOINT_NAME]["metric_name"],

    "best_metric_value":
        CHECKPOINT_STATES[LIVE_CHECKPOINT_NAME]["best_metric"],

    "best_epoch":
        CHECKPOINT_STATES[LIVE_CHECKPOINT_NAME]["best_epoch"],

    "higher_is_better":
        CHECKPOINT_STATES[LIVE_CHECKPOINT_NAME]["higher_is_better"],

    "final_gate_weights": {

    "lstm_weight":
        float(live_model.last_lstm_weight.cpu())
        if hasattr(live_model, "last_lstm_weight")
        else None,

    "mlp_weight":
        float(live_model.last_mlp_weight.cpu())
        if hasattr(live_model, "last_mlp_weight")
        else None,
    },


    # ========================================================
    # MODEL
    # ========================================================

    "model_state_dict":
        copy.deepcopy(live_model.state_dict()),

    "model_params": {

    "seq_input_size":
        saved_seq_input_size,

    "feature_input_size":
        saved_feature_input_size,

    "num_classes":
        saved_num_classes,

    "bidirectional":
        True,

    # CNN
    "conv1_out_channels":
        128,

    "conv1_kernel_size":
        15,

    "conv2_out_channels":
        256,

    "conv2_kernel_size":
        9,

    "maxpool_size":
        4,

    # LSTM
    "lstm_input_size":
        256,

    "lstm_hidden_size":
        64,

    "lstm_num_layers":
        1,

    # LSTM Projector
    "lstm_projector_hidden_1":
        32,

    "lstm_projector_hidden_2":
        8,

    # Feature MLP
    "feature_mlp_hidden_1":
        128,

    "feature_mlp_hidden_2":
        32,

    "feature_mlp_hidden_3":
        16,

    # Classifier
    "classifier_hidden_1":
        16,

    "classifier_hidden_2":
        4,

    # Fusion Gate
    "gate_input_size":
        24,

    "gate_output_size":
        2,
    },
    # ========================================================
    # INPUT INFO
    # ========================================================

    "seq_input_size":
        saved_seq_input_size,

    "feature_input_size":
        saved_feature_input_size,

    "num_classes":
        saved_num_classes,


    # ========================================================
    # PREPROCESSORS
    # ========================================================

    "seq_scaler":
        saved_seq_scaler,

    "feature_scaler":
        saved_feature_scaler,

    "csp":
        saved_csp,


    # ========================================================
    # FEATURE EXTRACTOR
    # ========================================================

    "extractor_params": {

        "c3_idx":
            saved_c3_idx,

        "cz_idx":
            saved_cz_idx,

        "c4_idx":
            saved_c4_idx,

        "fs":
            saved_sampling_rate,

        "bands":
            saved_bands,

        "n_csp_components":
            saved_n_csp_components,

        "csp_reg":
            saved_csp_reg,
    },

    "extractor_state": {

        "channel_names":
            saved_channel_names,

        "feature_names":
            saved_feature_names,
    },


    # ========================================================
    # TRAINING HISTORY
    # ========================================================

    "history":
        history,


    # ========================================================
    # ALL BEST MODELS
    # ========================================================

    "all_best_states": {

        "lowest_val_loss": {
            "state_dict": best_val_loss_state,
            "best_metric": best_val_loss,
            "best_epoch": best_val_loss_epoch,
        },

        "highest_val_auc": {
            "state_dict": best_val_auc_state,
            "best_metric": best_val_auc,
            "best_epoch": best_val_auc_epoch,
        },

        "highest_val_acc": {
            "state_dict": best_val_acc_state,
            "best_metric": best_val_acc,
            "best_epoch": best_val_acc_epoch,
        },

        "highest_val_ulta_ratio": {
            "state_dict": best_val_ulta_ratio_state,
            "best_metric": best_val_ulta_ratio,
            "best_epoch": best_val_ulta_ratio_epoch,
        },
    },


    # ========================================================
    # LABEL MAPPING
    # ========================================================

    "class_to_command": {
        0: "left",
        1: "right",
    },
}


# ============================================================
# SAVE
# ============================================================

joblib.dump(
    checkpoint,
    MODEL_SAVE_PATH
)

print()
print("=" * 60)
print("Saved BrainDance checkpoint")
print("=" * 60)

print("Path:", MODEL_SAVE_PATH)
print()

print("Checkpoint:", LIVE_CHECKPOINT_NAME)
print("Metric:", checkpoint["best_metric_name"])
print("Value:", checkpoint["best_metric_value"])
print("Epoch:", checkpoint["best_epoch"])
print()

print("Seq input size:", saved_seq_input_size)
print("Feature input size:", saved_feature_input_size)
print("Num classes:", saved_num_classes)

print(
    "Channel count:",
    None if saved_channel_names is None
    else len(saved_channel_names)
)

print(
    "Feature count:",
    None if saved_feature_names is None
    else len(saved_feature_names)
)

print()
print("Checkpoint saved successfully.")

In [ ]:
import numpy as np

data = np.load("Datasets/Main.npz", allow_pickle=True)

X = data["X_raw"]
y = data["y"]

BAD_CHANNELS = [0, 1, 2, 6, 7]
X = np.delete(X, BAD_CHANNELS, axis=1)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
import joblib
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score,
    f1_score,
    confusion_matrix,
)

# ============================================================
# LOAD CHECKPOINT
# ============================================================

checkpoint = joblib.load(
    "Model/braindance.joblib"
)

# ============================================================
# BUILD MODEL
# ============================================================

model = BraindanceCNNLSTMFusion(
    seq_input_size=seq_input_size,
    feature_input_size=feature_input_size,
    num_classes=num_classes,
    bidirectional=True
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.eval()

print("Model loaded successfully.")

# ============================================================
# INFERENCE
# ============================================================

all_preds = []
all_probs = []
all_labels = []

lstm_gates = []
mlp_gates = []

with torch.no_grad():

    for X_seq_batch, X_feat_batch, y_batch in test_loader:

        logits = model(
            X_seq_batch,
            X_feat_batch
        )

        probs = torch.softmax(
            logits,
            dim=1
        )

        preds = torch.argmax(
            probs,
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_probs.extend(
            probs[:, 1].cpu().numpy()
        )

        all_labels.extend(
            y_batch.cpu().numpy()
        )

        lstm_gates.append(
            float(model.last_lstm_weight.cpu())
        )

        mlp_gates.append(
            float(model.last_mlp_weight.cpu())
        )

# ============================================================
# METRICS
# ============================================================

acc = accuracy_score(
    all_labels,
    all_preds
)

bal_acc = balanced_accuracy_score(
    all_labels,
    all_preds
)

auc = roc_auc_score(
    all_labels,
    all_probs
)

f1 = f1_score(
    all_labels,
    all_preds
)

cm = confusion_matrix(
    all_labels,
    all_preds
)

# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 60)
print("TEST RESULTS")
print("=" * 60)

print(f"Accuracy      : {acc:.4f}")
print(f"Balanced Acc  : {bal_acc:.4f}")
print(f"AUC           : {auc:.4f}")
print(f"F1 Score      : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nAverage Gate Weights:")
print(
    f"LSTM Gate : {np.mean(lstm_gates):.3f}"
)

print(
    f"MLP Gate  : {np.mean(mlp_gates):.3f}"
)